# 04 - Create Features

This notebook is in charge of creating additional variables from the cleaned weekly data.

These new variables will provide additional information on recent behaviour in financial stress, bond spreads, exchange rates and monetary policy before they are used in the modelling stage.

In [5]:
import pandas as pd
from pathlib import Path

In [6]:
data_dir = Path.cwd() .joinpath("swiss-bank-risk-project", "data")

processed_data_folder = data_dir.joinpath("processed")
processed_data_folder.mkdir(parents=True, exist_ok=True)

clean_weekly_data = processed_data_folder.joinpath("clean_weekly_data.csv")

clean_weekly_df = pd.read_csv(clean_weekly_data, parse_dates=["date"])

print(processed_data_folder)
print(clean_weekly_data)
display(clean_weekly_df.head())

/files/swiss-bank-risk-project/data/processed
/files/swiss-bank-risk-project/data/processed/clean_weekly_data.csv


,date,ciss_index,chf-eur_exchange_rate_control,commercial_bank_bond_yield,government_bond_yield,commercial_bank_bond_spread,snb_policy_rate
0,2010-01-08,0.165886,1.48398,2.1872,1.8552,0.3320,0.375
1,2010-01-15,0.145988,1.47692,2.1640,1.8228,0.3412,0.375
2,2010-01-22,0.186101,1.47398,2.1208,1.7722,0.3486,0.375
3,2010-01-29,0.245768,1.47118,2.1194,1.7716,0.3478,0.375
4,2010-02-05,0.229882,1.47136,2.0718,1.7392,0.3326,0.375


## Changes in the main variables

It's important to check how the main financial variables move on a week-to-week basis. Changes are therefore computed for each of them.

This is of particular relevance because the aim of the project is to assess whether changes in euro area financial stress can help predict changes in perceived bank risk.

In [7]:
ciss_change = clean_weekly_df["ciss_index"].diff()
bond_spread_change = clean_weekly_df["commercial_bank_bond_spread"].diff()
commercial_bond_yield_change = clean_weekly_df["commercial_bank_bond_yield"].diff()
government_bond_yield_change = clean_weekly_df["government_bond_yield"].diff()
policy_rate_change = clean_weekly_df["snb_policy_rate"].diff()

clean_weekly_df.insert(7, "ciss_change", ciss_change)
clean_weekly_df.insert(8, "bond_spread_change", bond_spread_change)
clean_weekly_df.insert(9, "commercial_bond_yield_change", commercial_bond_yield_change)
clean_weekly_df.insert(10, "government_bond_yield_change", government_bond_yield_change)
clean_weekly_df.insert(11, "policy_rate_change", policy_rate_change)

display(clean_weekly_df.head())

,date,ciss_index,chf-eur_exchange_rate_control,commercial_bank_bond_yield,government_bond_yield,commercial_bank_bond_spread,snb_policy_rate,ciss_change,bond_spread_change,commercial_bond_yield_change,government_bond_yield_change,policy_rate_change
0,2010-01-08,0.165886,1.48398,2.1872,1.8552,0.3320,0.375,NaN,NaN,NaN,NaN,NaN
1,2010-01-15,0.145988,1.47692,2.1640,1.8228,0.3412,0.375,-0.019898,0.0092,-0.0232,-0.0324,0.0
2,2010-01-22,0.186101,1.47398,2.1208,1.7722,0.3486,0.375,0.040113,0.0074,-0.0432,-0.0506,0.0
3,2010-01-29,0.245768,1.47118,2.1194,1.7716,0.3478,0.375,0.059668,-0.0008,-0.0014,-0.0006,0.0
4,2010-02-05,0.229882,1.47136,2.0718,1.7392,0.3326,0.375,-0.015887,-0.0152,-0.0476,-0.0324,0.0


## Volatility measures

Rolling volatility measures (volatility over a recent time period) are created to capture how unstable some variables have been over the previous weeks.

Higher volatilities may indicate periods of greater uncertainty or more extreme movements in financial markets.

In [8]:
ciss_volatility_past_4w = clean_weekly_df["ciss_index"].rolling(window=4).std()
bond_spread_volatility_past_4w = clean_weekly_df["commercial_bank_bond_spread"].rolling(window=4).std()
exchange_rate_volatility_past_4w = clean_weekly_df["chf-eur_exchange_rate_control"].rolling(window=4).std()

clean_weekly_df.insert(12, "ciss_volatility_past_4_weeks", ciss_volatility_past_4w)
clean_weekly_df.insert(13, "bond_spread_volatility_past_4_weeks", bond_spread_volatility_past_4w)
clean_weekly_df.insert(14, "exchange_rate_volatility_past_4_weeks", exchange_rate_volatility_past_4w)

display(clean_weekly_df.head())

,date,ciss_index,chf-eur_exchange_rate_control,commercial_bank_bond_yield,government_bond_yield,commercial_bank_bond_spread,snb_policy_rate,ciss_change,bond_spread_change,commercial_bond_yield_change,government_bond_yield_change,policy_rate_change,ciss_volatility_past_4_weeks,bond_spread_volatility_past_4_weeks,exchange_rate_volatility_past_4_weeks
0,2010-01-08,0.165886,1.48398,2.1872,1.8552,0.3320,0.375,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-15,0.145988,1.47692,2.1640,1.8228,0.3412,0.375,-0.019898,0.0092,-0.0232,-0.0324,0.0,NaN,NaN,NaN
2,2010-01-22,0.186101,1.47398,2.1208,1.7722,0.3486,0.375,0.040113,0.0074,-0.0432,-0.0506,0.0,NaN,NaN,NaN
3,2010-01-29,0.245768,1.47118,2.1194,1.7716,0.3478,0.375,0.059668,-0.0008,-0.0014,-0.0006,0.0,0.043119,0.007685,0.005501
4,2010-02-05,0.229882,1.47136,2.0718,1.7392,0.3326,0.375,-0.015887,-0.0152,-0.0476,-0.0324,0.0,0.045030,0.007416,0.002696


## Lagged variables

Lagged variables are created to include information from previous weeks.

This allows the models to use past values of financial stress and bank bond spreads when predicting future changes in perceived bank risk.

In [9]:
clean_weekly_df["ciss_lag_1w"] = clean_weekly_df["ciss_index"].shift(1)
clean_weekly_df["ciss_lag_4w"] = clean_weekly_df["ciss_index"].shift(4)
clean_weekly_df["bond_spread_lag_1w"] = clean_weekly_df["commercial_bank_bond_spread"].shift(1)
clean_weekly_df["bond_spread_lag_4w"] = clean_weekly_df["commercial_bank_bond_spread"].shift(4)

display(clean_weekly_df.head())

,date,ciss_index,chf-eur_exchange_rate_control,commercial_bank_bond_yield,government_bond_yield,commercial_bank_bond_spread,snb_policy_rate,ciss_change,bond_spread_change,commercial_bond_yield_change,government_bond_yield_change,policy_rate_change,ciss_volatility_past_4_weeks,bond_spread_volatility_past_4_weeks,exchange_rate_volatility_past_4_weeks,ciss_lag_1w,ciss_lag_4w,bond_spread_lag_1w,bond_spread_lag_4w
0,2010-01-08,0.165886,1.48398,2.1872,1.8552,0.3320,0.375,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-15,0.145988,1.47692,2.1640,1.8228,0.3412,0.375,-0.019898,0.0092,-0.0232,-0.0324,0.0,NaN,NaN,NaN,0.165886,NaN,0.3320,NaN
2,2010-01-22,0.186101,1.47398,2.1208,1.7722,0.3486,0.375,0.040113,0.0074,-0.0432,-0.0506,0.0,NaN,NaN,NaN,0.145988,NaN,0.3412,NaN
3,2010-01-29,0.245768,1.47118,2.1194,1.7716,0.3478,0.375,0.059668,-0.0008,-0.0014,-0.0006,0.0,0.043119,0.007685,0.005501,0.186101,NaN,0.3486,NaN
4,2010-02-05,0.229882,1.47136,2.0718,1.7392,0.3326,0.375,-0.015887,-0.0152,-0.0476,-0.0324,0.0,0.045030,0.007416,0.002696,0.245768,0.165886,0.3478,0.332


## Checking the created features

The newly created lagged features will have some missing values at the beginning of the data set given that they rely on previous weeks.

There should be 1 missing value for a 1-week lagged variable and 4 missing values for a 4-week lagged variable.

In [10]:
print("Missing values:")
print(clean_weekly_df.isna().sum())

missing_rows = clean_weekly_df[clean_weekly_df.isna().any(axis=1)]

print("Rows with missing values:")
display(missing_rows)

Missing values:
date                                     0
ciss_index                               0
chf-eur_exchange_rate_control            0
commercial_bank_bond_yield               0
government_bond_yield                    0
commercial_bank_bond_spread              0
snb_policy_rate                          0
ciss_change                              1
bond_spread_change                       1
commercial_bond_yield_change             1
government_bond_yield_change             1
policy_rate_change                       1
ciss_volatility_past_4_weeks             3
bond_spread_volatility_past_4_weeks      3
exchange_rate_volatility_past_4_weeks    3
ciss_lag_1w                              1
ciss_lag_4w                              4
bond_spread_lag_1w                       1
bond_spread_lag_4w                       4
dtype: int64
Rows with missing values:


,date,ciss_index,chf-eur_exchange_rate_control,commercial_bank_bond_yield,government_bond_yield,commercial_bank_bond_spread,snb_policy_rate,ciss_change,bond_spread_change,commercial_bond_yield_change,government_bond_yield_change,policy_rate_change,ciss_volatility_past_4_weeks,bond_spread_volatility_past_4_weeks,exchange_rate_volatility_past_4_weeks,ciss_lag_1w,ciss_lag_4w,bond_spread_lag_1w,bond_spread_lag_4w
0,2010-01-08,0.165886,1.48398,2.1872,1.8552,0.3320,0.375,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-15,0.145988,1.47692,2.1640,1.8228,0.3412,0.375,-0.019898,0.0092,-0.0232,-0.0324,0.0,NaN,NaN,NaN,0.165886,NaN,0.3320,NaN
2,2010-01-22,0.186101,1.47398,2.1208,1.7722,0.3486,0.375,0.040113,0.0074,-0.0432,-0.0506,0.0,NaN,NaN,NaN,0.145988,NaN,0.3412,NaN
3,2010-01-29,0.245768,1.47118,2.1194,1.7716,0.3478,0.375,0.059668,-0.0008,-0.0014,-0.0006,0.0,0.043119,0.007685,0.005501,0.186101,NaN,0.3486,NaN


## Output

The new features have now been added to the weekly data set. The final data is checked before being saved for use in the modelling stage.

The final data set is now saved as `feature_weekly_data.csv` in the `data/processed` folder.

In [11]:
print("Missing values:")
print(clean_weekly_df.isna().sum())


print("Duplicate dates:")
print(clean_weekly_df["date"].duplicated().sum())


print("Final shape:")
print(clean_weekly_df.shape)


print("Date range:")
print(clean_weekly_df["date"].min())
print(clean_weekly_df["date"].max())


print("First rows:")
display(clean_weekly_df.head())


print("Last rows:")
display(clean_weekly_df.tail())


output_file = processed_data_folder.joinpath("feature_weekly_data.csv") # we make the path


clean_weekly_df.to_csv(output_file, index=False) # this is the function that saves the table to a new csv file


print("Saved feature dataset to:")
print(output_file)

Missing values:
date                                     0
ciss_index                               0
chf-eur_exchange_rate_control            0
commercial_bank_bond_yield               0
government_bond_yield                    0
commercial_bank_bond_spread              0
snb_policy_rate                          0
ciss_change                              1
bond_spread_change                       1
commercial_bond_yield_change             1
government_bond_yield_change             1
policy_rate_change                       1
ciss_volatility_past_4_weeks             3
bond_spread_volatility_past_4_weeks      3
exchange_rate_volatility_past_4_weeks    3
ciss_lag_1w                              1
ciss_lag_4w                              4
bond_spread_lag_1w                       1
bond_spread_lag_4w                       4
dtype: int64
Duplicate dates:
0
Final shape:
(813, 19)
Date range:
2010-01-08 00:00:00
2025-08-01 00:00:00
First rows:


,date,ciss_index,chf-eur_exchange_rate_control,commercial_bank_bond_yield,government_bond_yield,commercial_bank_bond_spread,snb_policy_rate,ciss_change,bond_spread_change,commercial_bond_yield_change,government_bond_yield_change,policy_rate_change,ciss_volatility_past_4_weeks,bond_spread_volatility_past_4_weeks,exchange_rate_volatility_past_4_weeks,ciss_lag_1w,ciss_lag_4w,bond_spread_lag_1w,bond_spread_lag_4w
0,2010-01-08,0.165886,1.48398,2.1872,1.8552,0.3320,0.375,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-15,0.145988,1.47692,2.1640,1.8228,0.3412,0.375,-0.019898,0.0092,-0.0232,-0.0324,0.0,NaN,NaN,NaN,0.165886,NaN,0.3320,NaN
2,2010-01-22,0.186101,1.47398,2.1208,1.7722,0.3486,0.375,0.040113,0.0074,-0.0432,-0.0506,0.0,NaN,NaN,NaN,0.145988,NaN,0.3412,NaN
3,2010-01-29,0.245768,1.47118,2.1194,1.7716,0.3478,0.375,0.059668,-0.0008,-0.0014,-0.0006,0.0,0.043119,0.007685,0.005501,0.186101,NaN,0.3486,NaN
4,2010-02-05,0.229882,1.47136,2.0718,1.7392,0.3326,0.375,-0.015887,-0.0152,-0.0476,-0.0324,0.0,0.045030,0.007416,0.002696,0.245768,0.165886,0.3478,0.332


Last rows:


,date,ciss_index,chf-eur_exchange_rate_control,commercial_bank_bond_yield,government_bond_yield,commercial_bank_bond_spread,snb_policy_rate,ciss_change,bond_spread_change,commercial_bond_yield_change,government_bond_yield_change,policy_rate_change,ciss_volatility_past_4_weeks,bond_spread_volatility_past_4_weeks,exchange_rate_volatility_past_4_weeks,ciss_lag_1w,ciss_lag_4w,bond_spread_lag_1w,bond_spread_lag_4w
808,2025-07-04,0.030363,0.933960,0.9386,0.2976,0.6410,0.0,0.006354,0.0020,0.0152,0.0132,0.0,0.003906,0.008776,0.002678,0.024009,0.015220,0.6390,0.62660
809,2025-07-11,0.028096,0.933200,0.9658,0.3344,0.6314,0.0,-0.002266,-0.0096,0.0272,0.0368,0.0,0.002924,0.008711,0.003213,0.030363,0.022962,0.6410,0.63725
810,2025-07-18,0.030874,0.931580,0.9958,0.3546,0.6412,0.0,0.002778,0.0098,0.0300,0.0202,0.0,0.003127,0.004608,0.002481,0.028096,0.030051,0.6314,0.62180
811,2025-07-25,0.032790,0.932820,0.9622,0.3262,0.6360,0.0,0.001916,-0.0052,-0.0336,-0.0284,0.0,0.001930,0.004668,0.000994,0.030874,0.024009,0.6412,0.63900
812,2025-08-01,0.025498,0.930925,0.9225,0.2835,0.6390,0.0,-0.007292,0.0030,-0.0397,-0.0427,0.0,0.003192,0.004241,0.001061,0.032790,0.030363,0.6360,0.64100


Saved feature dataset to:
/files/swiss-bank-risk-project/data/processed/feature_weekly_data.csv


## Summary

The weekly data has now been updated to include changes, volatility measures and lagged variables.

The final feature data set is now ready for the modelling stage.